# SAP RPT-1 via Python `requests`

This notebook is a complete tutorial for consuming **SAP RPT-1** from SAP AI Core using plain HTTP requests.

It covers:
- Authentication (`OAuth2 client_credentials`)
- Model discovery
- Configuration and deployment creation
- Prediction requests (`rows` and `columns` payload styles)
- Response interpretation
- Optional cleanup

> The workflow is aligned with the SAP AI Core foundation-models scenario and the `aicore-sap` executable used for `sap-rpt-1-small` / `sap-rpt-1-large`.


## 1) Prerequisites

### Python packages
```bash
pip install requests python-dotenv
```

### Environment variables
Set these values in `.env` (or your shell environment):

```bash
AICORE_BASE_URL=<YOUR_AI_API_URL>              # with or without /v2
AICORE_AUTH_URL=<YOUR_AUTH_URL>
AICORE_CLIENT_ID=<YOUR_CLIENT_ID>
AICORE_CLIENT_SECRET=<YOUR_CLIENT_SECRET>
AICORE_RESOURCE_GROUP=default

# Optional: reuse an existing deployment instead of creating one in this notebook
RPT1_DEPLOYMENT_URL=<YOUR_EXISTING_DEPLOYMENT_URL>
RPT1_DEPLOYMENT_ID=<YOUR_EXISTING_DEPLOYMENT_ID>
```

### Notes
- This notebook can create billable resources (configuration + deployment) if `RPT1_DEPLOYMENT_URL` is not set.
- Keep one resource group consistently across setup, deployment, and prediction calls.


In [1]:
import json
import os
import time
import uuid
from typing import Any

import requests
from dotenv import load_dotenv


In [9]:
load_dotenv()


def normalize_api_base_url(url: str) -> str:
    """Normalize API base URL so endpoints can always append '/v2/...'."""
    if not url:
        return url
    normalized = url.strip().rstrip('/')
    if normalized.endswith('/v2'):
        normalized = normalized[:-3]
    return normalized


def mask(value: str | None, keep: int = 4) -> str:
    if not value:
        return '<missing>'
    if len(value) <= keep:
        return '*' * len(value)
    return ('*' * (len(value) - keep)) + value[-keep:]


AICORE_BASE_URL = normalize_api_base_url(os.getenv('AICORE_BASE_URL', ''))
AICORE_AUTH_URL = os.getenv('AICORE_AUTH_URL', '').rstrip('/')
AICORE_CLIENT_ID = os.getenv('AICORE_CLIENT_ID', '')
AICORE_CLIENT_SECRET = os.getenv('AICORE_CLIENT_SECRET', '')
AICORE_RESOURCE_GROUP = os.getenv('AICORE_RESOURCE_GROUP', 'default')

RPT1_DEPLOYMENT_URL = os.getenv('RPT1_DEPLOYMENT_URL', '').rstrip('/')
RPT1_DEPLOYMENT_ID = os.getenv('RPT1_DEPLOYMENT_ID', '').strip()

required = {
    'AICORE_BASE_URL': AICORE_BASE_URL,
    'AICORE_AUTH_URL': AICORE_AUTH_URL,
    'AICORE_CLIENT_ID': AICORE_CLIENT_ID,
    'AICORE_CLIENT_SECRET': AICORE_CLIENT_SECRET,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f'Missing required environment variables: {missing}')

print('Environment is correct:')
print(
    json.dumps(
        {
            'AICORE_BASE_URL': AICORE_BASE_URL,
            'AICORE_AUTH_URL': AICORE_AUTH_URL,
            'AICORE_CLIENT_ID': mask(AICORE_CLIENT_ID),
            'AICORE_CLIENT_SECRET': mask(AICORE_CLIENT_SECRET),
            'AICORE_RESOURCE_GROUP': AICORE_RESOURCE_GROUP,
            'RPT1_DEPLOYMENT_URL': mask(RPT1_DEPLOYMENT_URL) or '<not set>',
            'RPT1_DEPLOYMENT_ID': mask(RPT1_DEPLOYMENT_ID) or '<not set>',
        },
        indent=2,
    )
)


Environment is correct:
{
  "AICORE_BASE_URL": "https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com",
  "AICORE_AUTH_URL": "https://dts-ai-core.authentication.eu10.hana.ondemand.com",
  "AICORE_CLIENT_ID": "*******************************************************b540",
  "AICORE_CLIENT_SECRET": "*****************************************************************************jr0=",
  "AICORE_RESOURCE_GROUP": "default",
  "RPT1_DEPLOYMENT_URL": "***********************************************************************************************c4b1",
  "RPT1_DEPLOYMENT_ID": "************c4b1"
}


## 2) Helper functions for authenticated API calls


In [3]:
session = requests.Session()


def check_response(response: requests.Response) -> None:
    """Raise a detailed exception for non-2xx responses."""
    if response.ok:
        return
    message = (
        f"HTTP {response.status_code} for {response.request.method} {response.url}\n"
        f"Response: {response.text[:2000]}"
    )
    raise requests.HTTPError(message, response=response)


def request_json(
    method: str,
    url: str,
    *,
    headers: dict[str, str],
    payload: dict[str, Any] | None = None,
    params: dict[str, Any] | None = None,
    timeout: int = 120,
) -> dict[str, Any]:
    response = session.request(
        method=method,
        url=url,
        headers=headers,
        json=payload,
        params=params,
        timeout=timeout,
    )
    check_response(response)
    if response.text.strip():
        return response.json()
    return {}


def get_access_token() -> str:
    token_url = f"{AICORE_AUTH_URL}/oauth/token"
    response = session.post(
        token_url,
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        data={
            'grant_type': 'client_credentials',
            'client_id': AICORE_CLIENT_ID,
            'client_secret': AICORE_CLIENT_SECRET,
        },
        timeout=60,
    )
    check_response(response)
    token = response.json().get('access_token')
    if not token:
        raise RuntimeError(f'No access_token in auth response: {response.text[:1000]}')
    return token


def build_headers(token: str) -> dict[str, str]:
    return {
        'Authorization': f'Bearer {token}',
        'AI-Resource-Group': AICORE_RESOURCE_GROUP,
        'Content-Type': 'application/json',
    }


## 3) Authenticate and verify access


In [4]:
access_token = get_access_token()
headers = build_headers(access_token)

print(f'Access token acquired (length={len(access_token)}).')

# Optional sanity check: confirm the foundation-models scenario is visible.
scenarios = request_json('GET', f'{AICORE_BASE_URL}/v2/lm/scenarios', headers=headers)
scenario_ids = [resource.get('id') for resource in scenarios.get('resources', [])]

print(f'Scenarios available: {len(scenario_ids)}')
print('Contains foundation-models:', 'foundation-models' in scenario_ids)


Access token acquired (length=7270).
Scenarios available: 12
Contains foundation-models: True


## 4) Discover available foundation models and find RPT-1

According to SAP documentation, RPT-1 is available under:
- `executableId`: `aicore-sap`
- `model`: `sap-rpt-1-small` or `sap-rpt-1-large`


In [5]:
models_response = request_json(
    'GET',
    f'{AICORE_BASE_URL}/v2/lm/scenarios/foundation-models/models',
    headers=headers,
)

resources = models_response.get('resources', [])
rpt_models = [m for m in resources if str(m.get('model', '')).startswith('sap-rpt-1')]

print(f'Total foundation models visible: {len(resources)}')
print(f'RPT-1 models visible: {len(rpt_models)}')

for model in rpt_models:
    versions = model.get('versions', [])
    version_names = [v.get('name') for v in versions if v.get('name')]
    latest_versions = [v.get('name') for v in versions if v.get('isLatest')]
    print('-' * 80)
    print('model         :', model.get('model'))
    print('executableId  :', model.get('executableId'))
    print('provider      :', model.get('provider'))
    print('latest version:', latest_versions[0] if latest_versions else '<not marked>')
    print('versions      :', ', '.join(version_names[:10]) if version_names else '<none listed>')


Total foundation models visible: 46
RPT-1 models visible: 2
--------------------------------------------------------------------------------
model         : sap-rpt-1-large
executableId  : aicore-sap
provider      : SAP
latest version: 1
versions      : 1
--------------------------------------------------------------------------------
model         : sap-rpt-1-small
executableId  : aicore-sap
provider      : SAP
latest version: 1
versions      : 1


## 5) Choose model settings

Default choices below are safe for getting started:
- `sap-rpt-1-small` for lower latency
- `latest` model version for auto-upgrade behavior

Change these values if you need `sap-rpt-1-large` or a pinned version.


In [10]:
EXECUTABLE_ID = 'aicore-sap'
SCENARIO_ID = 'foundation-models'
MODEL_NAME = os.getenv('RPT1_MODEL_NAME', 'sap-rpt-1-small')
MODEL_VERSION = os.getenv('RPT1_MODEL_VERSION', 'latest')
VERSION_ID = '1.0.0'
NAME_PREFIX = os.getenv('RPT1_NAME_PREFIX', 'rpt1-requests-tutorial')

print(
    json.dumps(
        {
            'EXECUTABLE_ID': EXECUTABLE_ID,
            'SCENARIO_ID': SCENARIO_ID,
            'MODEL_NAME': MODEL_NAME,
            'MODEL_VERSION': MODEL_VERSION,
            'VERSION_ID': VERSION_ID,
            'NAME_PREFIX': NAME_PREFIX,
        },
        indent=2,
    )
)


{
  "EXECUTABLE_ID": "aicore-sap",
  "SCENARIO_ID": "foundation-models",
  "MODEL_NAME": "sap-rpt-1-small",
  "MODEL_VERSION": "latest",
  "VERSION_ID": "1.0.0",
  "NAME_PREFIX": "rpt1-requests-tutorial"
}


## 6) Create configuration and deployment (or reuse existing deployment)

The easiest way to create a deployment for a model is through the AI Core web Interface as shown in [this tutorial](https://developers.sap.com/tutorials/ai-core-generative-ai.html#6c4a539e-2bdf-4ddb-97a0-0f8d0f1bd00e). Alternatively, if an RPT-1 Deployement information is not added in the `.env` environment file, the following code can create this deployment.

If `RPT1_DEPLOYMENT_URL` is set, this notebook skips deployment creation and uses that URL.
Otherwise it will:
1. Create a configuration at `/v2/lm/configurations`
2. Create a deployment at `/v2/lm/deployments`
3. Poll deployment status until ready


In [15]:
def create_configuration() -> tuple[str, dict[str, Any], dict[str, Any]]:
    config_name = f"{NAME_PREFIX}-cfg-{uuid.uuid4().hex[:8]}"
    payload = {
        'name': config_name,
        'executableId': EXECUTABLE_ID,
        'scenarioId': SCENARIO_ID,
        'versionId': VERSION_ID,
        'parameterBindings': [
            {'key': 'modelName', 'value': MODEL_NAME},
            {'key': 'modelVersion', 'value': MODEL_VERSION},
        ],
    }

    response = request_json(
        'POST',
        f'{AICORE_BASE_URL}/v2/lm/configurations',
        headers=headers,
        payload=payload,
    )

    configuration_id = response.get('id') or response.get('configurationId')
    if not configuration_id:
        raise RuntimeError(f'Could not find configuration ID in response: {response}')

    return configuration_id, payload, response


def create_deployment(configuration_id: str) -> tuple[str, dict[str, Any]]:
    response = request_json(
        'POST',
        f'{AICORE_BASE_URL}/v2/lm/deployments',
        headers=headers,
        payload={'configurationId': configuration_id},
    )
    deployment_id = response.get('id') or response.get('deploymentId')
    if not deployment_id:
        raise RuntimeError(f'Could not find deployment ID in response: {response}')
    return deployment_id, response


def get_deployment(deployment_id: str) -> dict[str, Any]:
    return request_json(
        'GET',
        f'{AICORE_BASE_URL}/v2/lm/deployments/{deployment_id}',
        headers=headers,
    )


def deployment_state(payload: dict[str, Any]) -> str:
    # API variants can differ; this checks common status fields.
    candidates = [
        payload.get('status'),
        payload.get('targetStatus'),
        payload.get('state'),
        payload.get('lastOperation'),
    ]
    for value in candidates:
        if isinstance(value, str) and value:
            return value.upper()
        if isinstance(value, dict):
            nested = value.get('status') or value.get('state') or value.get('name')
            if isinstance(nested, str) and nested:
                return nested.upper()
    return 'UNKNOWN'


def wait_for_deployment_ready(
    deployment_id: str,
    timeout_seconds: int = 900,
    poll_interval_seconds: int = 15,
) -> dict[str, Any]:
    success_states = {'RUNNING', 'READY', 'COMPLETED'}
    failure_states = {'FAILED', 'ERROR', 'STOPPED', 'DELETED'}

    start = time.time()
    while True:
        details = get_deployment(deployment_id)
        state = deployment_state(details)
        print(f'deployment={deployment_id} state={state}')

        if state in success_states and details.get('deploymentUrl'):
            return details

        if state in failure_states:
            raise RuntimeError(f'Deployment entered failure state: {state}. Details: {details}')

        if time.time() - start > timeout_seconds:
            raise TimeoutError(f'Deployment not ready after {timeout_seconds}s. Last details: {details}')

        time.sleep(poll_interval_seconds)


configuration_id = None
deployment_id = RPT1_DEPLOYMENT_ID or None

if RPT1_DEPLOYMENT_URL:
    DEPLOYMENT_URL = RPT1_DEPLOYMENT_URL
    print('Using existing deployment URL from environment:')
    print(mask(DEPLOYMENT_URL))
else:
    configuration_id, configuration_payload, configuration_response = create_configuration()
    print('Created configuration:', mask(configuration_id))

    deployment_id, deployment_response = create_deployment(configuration_id)
    print('Created deployment:', mask(deployment_id))

    deployment_details = wait_for_deployment_ready(deployment_id)
    DEPLOYMENT_URL = deployment_details.get('deploymentUrl', '').rstrip('/')

if not DEPLOYMENT_URL:
    raise RuntimeError('Deployment URL is missing. Set RPT1_DEPLOYMENT_URL or create a deployment first.')

print('Deployment URL ready:')
print(mask(DEPLOYMENT_URL))


Using existing deployment URL from environment:
***********************************************************************************************c4b1
Deployment URL ready:
***********************************************************************************************c4b1


## 7) Predict using the **rows** payload format

In this format, each record is represented as a JSON object inside `rows`.
Use a placeholder such as `[PREDICT]` in the target column for rows you want predicted.


In [12]:
rows_payload = {
    'prediction_config': {
        'target_columns': [
            {
                'name': 'COSTCENTER',
                'prediction_placeholder': '[PREDICT]',
                'task_type': 'classification',
            }
        ]
    },
    'index_column': 'ID',
    'rows': [
        {
            'PRODUCT': 'Couch',
            'PRICE': 999.99,
            'ORDERDATE': '28-11-2025',
            'ID': '35',
            'COSTCENTER': '[PREDICT]',
        },
        {
            'PRODUCT': 'Office Chair',
            'PRICE': 150.80,
            'ORDERDATE': '02-11-2025',
            'ID': '44',
            'COSTCENTER': 'Office Furniture',
        },
        {
            'PRODUCT': 'Server Rack',
            'PRICE': 2200.00,
            'ORDERDATE': '01-11-2025',
            'ID': '104',
            'COSTCENTER': 'Data Infrastructure',
        },
    ],
    'data_schema': {
        'PRODUCT': {'dtype': 'string'},
        'PRICE': {'dtype': 'numeric'},
        'ORDERDATE': {'dtype': 'date'},
        'ID': {'dtype': 'string'},
        'COSTCENTER': {'dtype': 'string'},
    },
}

rows_prediction = request_json(
    'POST',
    f'{DEPLOYMENT_URL}/predict',
    headers=headers,
    payload=rows_payload,
)

print(json.dumps(rows_prediction, indent=2))


{
  "id": "a72e6d81-bd67-4ae8-b9d3-5db69775478f",
  "metadata": {
    "num_columns": 5,
    "num_predictions": 1,
    "num_query_rows": 1,
    "num_rows": 2
  },
  "predictions": [
    {
      "COSTCENTER": [
        {
          "confidence": 0.97,
          "prediction": "Office Furniture"
        }
      ],
      "ID": "35"
    }
  ],
  "status": {
    "code": 0,
    "message": "ok"
  }
}


## 8) Predict using the **columns** payload format

In this format, data is passed as arrays per column under `columns`.
Choose either `rows` or `columns` in one request.


In [13]:
columns_payload = {
    'prediction_config': {
        'target_columns': [
            {
                'name': 'COSTCENTER',
                'prediction_placeholder': '[PREDICT]',
                'task_type': 'classification',
            }
        ]
    },
    'index_column': 'ID',
    'columns': {
        'PRODUCT': ['Couch', 'Office Chair', 'Server Rack'],
        'PRICE': [999.99, 150.80, 2200.00],
        'ORDERDATE': ['28-11-2025', '02-11-2025', '01-11-2025'],
        'ID': ['35', '44', '104'],
        'COSTCENTER': ['[PREDICT]', 'Office Furniture', 'Data Infrastructure'],
    },
    'data_schema': {
        'PRODUCT': {'dtype': 'string'},
        'PRICE': {'dtype': 'numeric'},
        'ORDERDATE': {'dtype': 'date'},
        'ID': {'dtype': 'string'},
        'COSTCENTER': {'dtype': 'string'},
    },
}

columns_prediction = request_json(
    'POST',
    f'{DEPLOYMENT_URL}/predict',
    headers=headers,
    payload=columns_payload,
)

print(json.dumps(columns_prediction, indent=2))


{
  "id": "cc1e74ef-9547-4a78-af2c-e52c8fc10024",
  "metadata": {
    "num_columns": 5,
    "num_predictions": 1,
    "num_query_rows": 1,
    "num_rows": 2
  },
  "predictions": [
    {
      "COSTCENTER": [
        {
          "confidence": 0.97,
          "prediction": "Office Furniture"
        }
      ],
      "ID": "35"
    }
  ],
  "status": {
    "code": 0,
    "message": "ok"
  }
}


## 9) Interpret prediction responses

The response includes:
- `status.code` (`0`=OK, `1`=warning, `2`=invalid input, `3`=server error)
- `predictions` aligned with your query rows
- `metadata` (`num_rows`, `num_columns`, `num_predictions`, `num_query_rows`)


In [14]:
def summarize_prediction_response(response_json: dict[str, Any], label: str) -> None:
    print('=' * 100)
    print(label)

    status = response_json.get('status', {})
    print('request_id :', response_json.get('id'))
    print('status     :', status.get('code'), status.get('message'))
    print('metadata   :', response_json.get('metadata', {}))

    for i, row in enumerate(response_json.get('predictions', []), start=1):
        passthrough = {k: v for k, v in row.items() if not isinstance(v, list)}
        targets = {k: v for k, v in row.items() if isinstance(v, list)}
        print(f'prediction_row_{i}:')
        if passthrough:
            print('  passthrough:', passthrough)
        for target_name, values in targets.items():
            if values:
                print(f'  {target_name}: prediction={values[0].get("prediction")}, confidence={values[0].get("confidence")}')


summarize_prediction_response(rows_prediction, 'Rows payload response summary')
summarize_prediction_response(columns_prediction, 'Columns payload response summary')


Rows payload response summary
request_id : a72e6d81-bd67-4ae8-b9d3-5db69775478f
status     : 0 ok
metadata   : {'num_columns': 5, 'num_predictions': 1, 'num_query_rows': 1, 'num_rows': 2}
prediction_row_1:
  passthrough: {'ID': '35'}
  COSTCENTER: prediction=Office Furniture, confidence=0.97
Columns payload response summary
request_id : cc1e74ef-9547-4a78-af2c-e52c8fc10024
status     : 0 ok
metadata   : {'num_columns': 5, 'num_predictions': 1, 'num_query_rows': 1, 'num_rows': 2}
prediction_row_1:
  passthrough: {'ID': '35'}
  COSTCENTER: prediction=Office Furniture, confidence=0.97


## Troubleshooting quick guide

- `401/403`: verify token scope, service key, and resource group.
- `404` on `/predict`: deployment URL may be wrong or deployment not ready.
- `422`: payload shape/type issue; check `target_columns`, placeholder values, and `data_schema`.
- Low-quality predictions: improve context-row quality, data consistency, and target signal.

Best practice highlights for RPT-1:
- Keep columns meaningful and consistent.
- Provide representative context examples.
- Prefer explicit `data_schema` and `task_type` for stable results.
